## Initial Analysis

#### Loading Python Modules:

In [ ]:
import numpy as np
import random as rd
import matplotlib.pyplot as plt
from sys import argv # argv is a list in Python, which contains the command-line arguments passed to the script.
from class_state_vec import state_vector
from class_obs import obs_da
from class_da_sys import da_system

#### Loading Data from Nature Run:

In [ ]:
#-----------------------------------------------------------------------
# Read the Lorenz-96 Nature Run
#-----------------------------------------------------------------------
PathFigures = 'figures/'

#infile      = 'lorenz63_run-001.pkl'  # Qiwen's run
infile      = 'lorenz63_run.pkl'    # Diego's run

sv          = state_vector()  # create an instance of the state_vector class
sv          = sv.load(infile) # load the state vector from the file
sv.setFigurePath(PathFigures)

x_nature    = sv.getTrajectory() # get the trajectory of the nature run
xdim, maxit = np.shape(x_nature)

print('xdim = ', xdim)
#print('x_nature = ', x_nature)
print('shape of x_nature = ', np.shape(x_nature))

#
# Plot the nature run
#
sv.plotTrajectory(x_nature, 'Lorenz-63 State $x$ Natural Run', 'nature_run.png')

#### Loading Data from Observations:

In [ ]:
#-----------------------------------------------------------------------
# Read the Lorenz-96 observations
#-----------------------------------------------------------------------
infile = 'y_obs.pkl'
obs    = obs_da()
obs    = obs.load(infile)

#-----------------------------------------------------------------------
# Get the observations
#-----------------------------------------------------------------------
y_obs = obs.getVal()
y_pos = obs.getPos()
y_err = obs.getErr()

ydim, ycolumn = np.shape(y_obs)
#print('y_obs = ', y_obs)

#### Initialize the DA system:

In [ ]:
#-----------------------------------------------------------------------
# Create an instance of the da_system class
#-----------------------------------------------------------------------
das = da_system()         # create an instance of the da_system class
das.setStateVector(sv)    # store the state vector in the da_system class
das.setObsData(obs)       # store the observations in the da_system class
das.xdim = xdim
das.ydim = ydim
das.x0   = x_nature[:,0]  # initial condition
print('x_nature[:,0]: ', x_nature[:,0])
print(f'x_nature shape: {x_nature.shape}')
print('x_nature[:,105768]: ', x_nature[:,105768])

#### Initialize the Ensemble:

In [ ]:
#-----------------------------------------------------------------------
# Set the parameters to create the ensemble:
#-----------------------------------------------------------------------
das.edim           = 3
das.ens_bias_init  = 0
das.ens_sigma_init = np.sqrt(2)

#### Initialize the Error Covariances B and R, and the linearized Observation Operator H:

In [ ]:
I = np.identity(xdim)

#-----------------------------------------------------------------------
# Set Background Error Covariance
#-----------------------------------------------------------------------

B = 0 * I     # Background error covariance
das.setB(B)   # store the background error covariance in the da_system class

#-----------------------------------------------------------------------
# Set Observation Error Covariance
#-----------------------------------------------------------------------

R = 2.0 * I     # Observation error covariance
das.setR(R)   # store the observation error covariance in the da_system class

#-----------------------------------------------------------------------
# Set Linear Observation Operator
#-----------------------------------------------------------------------

H = I
das.setH(H)   # store the observation operator in the da_system


# Print the values of the state vector, observations, and the matrices B, R, and H
print('B = ')
print(das.getB())
print('R = ')
print(das.getR())
print('H = ')
print(das.getH())

#### Initialize the timesteps:

In [ ]:
#-----------------------------------------------------------------------
# Set time step and number of analysis steps:
#-----------------------------------------------------------------------
t_nature = sv.getTimes()
acyc_step = 8                               # how frequently to perform an analysis
dtau = (t_nature[acyc_step] - t_nature[0])
fcst_step = acyc_step                      # (may need to change for 4D DA methods)
fcst_dt = dtau / fcst_step


#-----------------------------------------------------------------------
# Store basic timing info in the da_system class
#-----------------------------------------------------------------------
das.acyc_step = acyc_step
das.dtau      = dtau
das.fcst_step = fcst_step
das.fcst_dt   = fcst_dt
das.dt        = (t_nature[1] - t_nature[0])
das.maxit     = maxit
das.xdim      = xdim


#print('t_nature = ', t_nature)
print('dtau = ', dtau)
print('fcst_dt = ', fcst_dt)
print('dt = ', das.dt)

#### Choose DA method:

In [ ]:
#-----------------------------------------------------------------------
# Ensemble methods
#-----------------------------------------------------------------------
# EnKF
method = 'EnKF'

# Particle filter
#method='PF'

#-----------------------------------------------------------------------
# 4D methods
#-----------------------------------------------------------------------
# 4D-Var
#method='4DVar'
# 4DEnVar
#method='4DEnVar'
# 4DETKF
#method='4DETKF'


das.setMethod(method)

#### Store DA Object:

In [ ]:
#-----------------------------------------------------------------------
# Store information in the da_system class
#-----------------------------------------------------------------------
name    = 'x_analysis_init'
outfile = name+'.pkl'
das.save(outfile)